In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, Trace
from pydantic import BaseModel
from agents.mcp import MCPServerStdio
import os

from IPython.display import display, Markdown, Image
import re

load_dotenv()

True

In [2]:
# # System Prompt Templates:

# agent1_instructions = ("You are a SQL Server analyst.\n"
#             "Workflow:\n"
#             "1) Use list_tables to find candidates.\n"
#             "2) Use describe_table before writing SQL.\n"
#             "3) Use execute_query for final SQL.\n"
#             "4) Return concise results and include the SQL used.")

# System Prompt Templates:

agent1_instructions = ("You are a database analyst creating executive-ready business summaries.\n"
            "Workflow:\n"
            "1) Mandatory discovery first: run execute_query on INFORMATION_SCHEMA.TABLES (BASE TABLE) and INFORMATION_SCHEMA.COLUMNS to find relevant tables/columns by keywords from the user request.\n"
            "2) Use list_tables as a secondary helper only; if it conflicts with SQL discovery, trust INFORMATION_SCHEMA results.\n"
            "3) Inspect top candidate tables with describe_table, then run minimal sample queries (for example SELECT TOP 1) to confirm semantics before final analysis SQL.\n"
            "4) Use execute_query for all analysis and final outputs.\n"
            "5) Use fetch only when external context is needed (for example definitions, benchmarks, market context, or supporting references).\n"
            "6) Adapt to the schema that exists in the current database; do not assume specific table or column names.\n"
            "7) If ideal fields are missing, proceed with best available proxies and clearly state assumptions and limitations instead of stopping.\n"
            "8) Do not ask the user to choose another database or ask permission to continue; proceed autonomously with the current data and provide best-effort findings.\n"
            "9) Do not claim data is unavailable until SQL discovery is complete and at least one relevant analysis query has been attempted.\n"
            "10) Accuracy rule: do not infer trends or rankings from unordered TOP queries; use explicit ORDER BY on the metric being analyzed.\n"
            "11) Output format: keep the main summary concise and business-friendly. Technical detail is allowed in a short 'Technical Notes' section.\n"
            "12) In the main summary, avoid SQL mechanics; include technical specifics only in 'Technical Notes' and 'SQL Appendix'.\n"
            "13) Include key insights, trends, implications, and notable data-quality caveats.\n"
            "14) Where helpful, use available chart tools to generate clear visuals to illustrate comparisons or trends; use text diagrams only if chart tools are unavailable.\n"
            "15) Add a final section titled 'SQL Appendix' and place the exact SQL query there.\n"
            "16) If fetch was used, add a short 'Sources' section with links.")

In [3]:
#MCP Params:
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

sql_params={
        "command": "mssql-mcp-server",   # or "npx", "args": ["-y", "mssql-mcp-server"]
        "env": {
            "MSSQL_CONNECTION_STRING": os.getenv("MSSQL_CONNECTION_STRING"),
            
            # "MSSQL_HOST": os.getenv("MSSQL_HOST"),
            # "MSSQL_PORT": os.getenv("MSSQL_PORT", "1433"),
            # "MSSQL_DATABASE": os.getenv("MSSQL_DATABASE"),
            # "MSSQL_USER": os.getenv("MSSQL_USER"),
            # "MSSQL_PASSWORD": os.getenv("MSSQL_PASSWORD"),
            # "MSSQL_ENCRYPT": os.getenv("MSSQL_ENCRYPT", "false"),
            # "MSSQL_TRUST_SERVER_CERTIFICATE": os.getenv("MSSQL_TRUST_SERVER_CERTIFICATE", "true"),            
        }
}

chart_params = {
      "command": "npx", "args": ["-y", "@antv/mcp-server-chart"]
    }
  

In [4]:
fetch_server = MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60)

mssql_server = MCPServerStdio(params=sql_params, client_session_timeout_seconds=60)

chart_server = MCPServerStdio(params=chart_params, client_session_timeout_seconds=60)

In [5]:
async with mssql_server:
    print(await mssql_server.call_tool("test_connection", {}))

meta=None content=[TextContent(type='text', text='[\n  {\n    "version": "Microsoft SQL Server 2019 (RTM) - 15.0.2000.5 (X64) \\n\\tSep 24 2019 13:48:23 \\n\\tCopyright (C) 2019 Microsoft Corporation\\n\\tExpress Edition (64-bit) on Windows 10 Home 10.0 <X64> (Build 26200: ) (Hypervisor)\\n",\n    "server_name": "LAPTOP-CPI3JAKQ\\\\SQLEXPRESS",\n    "database_name": "Certainty_Durham2"\n  }\n]', annotations=None, meta=None)] structuredContent=None isError=False


In [6]:
async with mssql_server:
    tools = await mssql_server.list_tools()
    print([t.name for t in tools])

['list_connections', 'test_connection', 'list_databases', 'list_tables', 'describe_table', 'sample_data', 'execute_query', 'get_relationships', 'list_stored_procedures', 'describe_stored_procedure', 'list_views', 'describe_view', 'list_triggers', 'describe_trigger', 'list_functions', 'list_indexes', 'analyze_table_stats', 'find_missing_indexes', 'analyze_index_usage', 'analyze_database_size', 'list_constraints', 'analyze_check_constraints', 'list_user_defined_types', 'find_computed_columns', 'list_default_constraints', 'analyze_data_distribution', 'find_lookup_tables', 'analyze_null_patterns', 'detect_audit_columns', 'get_stored_procedure_definition', 'get_multiple_stored_procedure_definitions', 'get_all_stored_procedure_definitions', 'search_stored_procedures_by_content']


In [7]:
async with fetch_server:
    tools = await fetch_server.list_tools()
    print([t.name for t in tools])

['fetch']


In [8]:
async with chart_server:
    tools = await chart_server.list_tools()
    print([t.name for t in tools])

['generate_area_chart', 'generate_bar_chart', 'generate_boxplot_chart', 'generate_column_chart', 'generate_district_map', 'generate_dual_axes_chart', 'generate_fishbone_diagram', 'generate_flow_diagram', 'generate_funnel_chart', 'generate_histogram_chart', 'generate_line_chart', 'generate_liquid_chart', 'generate_mind_map', 'generate_network_graph', 'generate_organization_chart', 'generate_path_map', 'generate_pie_chart', 'generate_pin_map', 'generate_radar_chart', 'generate_sankey_chart', 'generate_scatter_chart', 'generate_treemap_chart', 'generate_venn_chart', 'generate_violin_chart', 'generate_waterfall_chart', 'generate_word_cloud_chart', 'generate_spreadsheet']


In [ ]:
async with mssql_server, fetch_server, chart_server:
    agent = Agent(
            name="SQL Analyst",
            model="gpt-4o-mini",
            instructions=agent1_instructions,
            mcp_servers=[mssql_server, fetch_server, chart_server],
        )

    result = await Runner.run(
        agent,
        #"Show daily checklist completions for the last 30 days."
        #"Show daily checklist completions for the most recent 30 days that have data"
        "Find the latest CreationDate in dbo.CompletedChecklist, then show daily completion counts for the 30 days ending on that latest date. Generate the most appropriate chart if applicable. Only generate one chart."
    )
    #print(result.final_output)

    display(Markdown(result.final_output))  # renders markdown tables/images if present

    # optional: if output contains markdown image link, render it explicitly
    m = re.search(r'!\[[^\]]*\]\(([^)]+)\)', result.final_output or "")
    if m:
        display(Image(url=m.group(1)))

In [12]:
async with mssql_server, fetch_server, chart_server:
    agent = Agent(
            name="SQL Analyst",
            #model="gpt-4o-mini",
            model="gpt-5.2",
            instructions=agent1_instructions,
            mcp_servers=[mssql_server, fetch_server, chart_server],
        )

    result = await Runner.run(
        agent,
        #"Show daily checklist completions for the last 30 days."
        #"Show daily checklist completions for the most recent 30 days that have data"
        #"Which OCR checklist from what sites had the most negative incidents for the first month of 2025"
        "Which OCR checklist from what sites had the most negative incidents for the first month of 2025, use the following tables and more [dbo].[Checklist], [dbo].[ChecklistStructure], [dbo].[CompletedChecklist], [dbo].[CompletedChecklist_Answer], i think it might be in a Entity-Attribute-Value (EAV) "
        #"Identify the OCR checklist(s) with the highest number of negative incidents during January 1–31, 2025. Return: checklist name, site name, negative incident count, and rank (highest first). Include the SQL used and clearly state how you defined negative incident based on the schema."
        #"Top/bottom performing sites Rank sites by average checklist score in the last 90 days. Generate the most appropriate chart if applicable. Only generate one chart."
        #"Scan this dataabase and generate reports that you think will be useful. Generate the appropriate charts if applicable.", max_turns=60
        ,max_turns=50
    )
    #print(result.final_output)

    display(Markdown(result.final_output))  # renders markdown tables/images if present

    # optional: if output contains markdown image link, render it explicitly
    m = re.search(r'!\[[^\]]*\]\(([^)]+)\)', result.final_output or "")
    if m:
        display(Image(url=m.group(1)))

## Executive Summary (Jan 2025)
**The OCR checklist/site combination with the most negative incidents in the first month of 2025 was:**
- **Checklist:** *Job Site Safety Inspection Form (Foreman)*
- **Site:** *Regional Office 7*
- **Negative incidents:** **149**

This was the clear #1 driver of negative incidents for the month in the available data.

### Top checklist/site combinations (by negative incidents)
1. **Job Site Safety Inspection Form (Foreman)** — **Regional Office 7**: **149**
2. Job Site Safety Inspection Form (Inspection) — Regional Office 7: 55  
3. Job Site Safety Inspection Form (Inspection) — Research Facility 9: 54  
4. Job Site Safety Inspection Form (Inspection) — Executive Office 5: 45  
5. Job Site Safety Inspection Form (Foreman) — Research Facility 9: 40  
6. Job Site Safety Inspection Form (Inspection) — Data Center 4: 39  
7. Job Site Safety Inspection Form (Foreman) — Customer Support Office 12: 35  
8. Job Site Safety Inspection Form (Inspection) — Regional Office 8: 31  
9. Job Site Safety Inspection Form (Inspection) — Customer Support Office 10: 25  
10. Job Site Safety Inspection Form (Foreman) — Customer Support Office 10: 18  

### Visual (Top 10)
![Top OCR checklists/sites by negative incidents (Jan 2025)](https://mdn.alipayobjects.com/one_clip/afts/img/H78OSacimxgAAAAAU3AAAAgAoEACAQFr/original)

## Important caveat about “OCR”
Your database has OCR-related fields (e.g., `CompletedChecklist.OCRProcessedScanID`, `CompletedChecklist.OCRErrors`), **but for Jan 2025 there were 0 completed checklists with `OCRProcessedScanID` populated**.  
So the results above represent **negative incidents across completed checklists** in Jan 2025, **not OCR-ingested checklists specifically**. If your OCR indicator is stored elsewhere (or via an EAV attribute), it’s not reflected in `OCRProcessedScanID` for that month.

## Technical Notes (how “negative incidents” were counted)
Because this appears to be EAV-style (`CompletedChecklist_Answer` storing many answer rows per checklist), I treated a “negative incident” as any answer row that met **any** of:
- `CompletedChecklist_Answer.Score < 0` (explicit negative scoring), OR
- `AnswerValue` contains “Non Compliant”, OR
- `AnswerValue` = “No”

This aligns with the presence of answer options such as **“Non Compliant (- 1 … - 40)”**.

## SQL Appendix
```sql
-- Winner (top checklist/site) for Jan 2025
SELECT TOP 1 WITH TIES
  c.ChecklistName,
  s.SiteName,
  COUNT(*) AS NegativeIncidents
FROM dbo.CompletedChecklist cc
JOIN dbo.CompletedChecklist_Answer cca
  ON cca.CompletedChecklistID = cc.CompletedChecklistID
 AND cca.DeletionDate IS NULL
JOIN dbo.ChecklistStructure cs
  ON cs.ChecklistStructureID = cc.ChecklistStructureID
JOIN dbo.Checklist c
  ON c.ChecklistID = cs.ChecklistID
JOIN dbo.Site s
  ON s.SiteID = cc.SiteID
WHERE cc.CreationDate >= '2025-01-01'
  AND cc.CreationDate <  '2025-02-01'
  AND cc.DeletionDate IS NULL
  AND (
       cca.Score < 0
       OR cca.AnswerValue LIKE '%Non%Compliant%'
       OR cca.AnswerValue IN ('No')
  )
GROUP BY c.ChecklistName, s.SiteName
ORDER BY COUNT(*) DESC;

-- Top 20 list used for ranking/table
SELECT TOP 20
  c.ChecklistName,
  s.SiteName,
  COUNT(*) AS NegativeIncidents
FROM dbo.CompletedChecklist cc
JOIN dbo.CompletedChecklist_Answer cca
  ON cca.CompletedChecklistID = cc.CompletedChecklistID
 AND cca.DeletionDate IS NULL
JOIN dbo.ChecklistStructure cs
  ON cs.ChecklistStructureID = cc.ChecklistStructureID
JOIN dbo.Checklist c
  ON c.ChecklistID = cs.ChecklistID
JOIN dbo.Site s
  ON s.SiteID = cc.SiteID
WHERE cc.CreationDate >= '2025-01-01'
  AND cc.CreationDate <  '2025-02-01'
  AND cc.DeletionDate IS NULL
  AND (
       cca.Score < 0
       OR cca.AnswerValue LIKE '%Non%Compliant%'
       OR cca.AnswerValue IN ('No')
  )
GROUP BY c.ChecklistName, s.SiteName
ORDER BY COUNT(*) DESC, c.ChecklistName, s.SiteName;

-- OCR completeness check for Jan 2025 (shows OCRProcessedScanID unused that month)
SELECT
  COUNT(*) AS CompletedChecklists,
  SUM(CASE WHEN OCRProcessedScanID IS NOT NULL THEN 1 ELSE 0 END) AS OCRCompletedChecklists
FROM dbo.CompletedChecklist
WHERE CreationDate >= '2025-01-01'
  AND CreationDate <  '2025-02-01'
  AND DeletionDate IS NULL;
```

If you want, I can tighten the definition of “negative incidents” to **only scored negatives** (`Score < 0`) to avoid counting “No” answers that aren’t actually noncompliances—depends how your checklist logic is intended to work.